In [ ]:
from utils.orthanc_utils import *
from utils.db_utils import *

import pydicom as dicom
import numpy as np
import pydicom as dicom
import requests
import io
from tinydb import TinyDB, Query
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from pydicom.uid import generate_uid
from pydicom.uid import ExplicitVRLittleEndian
import pandas as pd


db = TinyDB('image_clasp_db.json')
StudyQuery = Query()
SeriesQuery = Query()
dl_processed_series = []

for study in db.search(StudyQuery.series.any(SeriesQuery.series_description == 'DL Processed')):
    study_id = study.get("orthanc_study_id")
    dicoms = []

    for series in study.get("series", []):
        if series.get("series_description") == "DL Processed":
            dl_processed_series.append({
                "study_id": study_id,
                "series": series
            })
            dicom_list = fetch_instances_for_series(series['orthanc_series_id'])
            dicoms.extend([fetch_dicom(dicom['ID']) for dicom in dicom_list])
            records = [{"SliceLocation": dcm.SliceLocation,
                        "InstanceNumber": dcm.InstanceNumber,
                        "PixelArray": dcm.pixel_array} for dcm in dicoms]
            df = pd.DataFrame(records).sort_values(["SliceLocation","InstanceNumber"])

            masks = []
            for slicelocation in df['SliceLocation'].unique():
                slice_df = df.loc[df['SliceLocation'] == slicelocation]
                time_masks = []
                for instancenumber in slice_df['InstanceNumber'].unique():
                    time_df = slice_df.loc[slice_df['InstanceNumber'] == instancenumber]
                    time_masks.append(time_df['PixelArray'].values[0])

                masks.append(np.stack(time_masks, -1))
            masks = np.stack(masks, -2)

            lv_mask = (masks == 500).astype(int)
            rv_mask = (masks == 1000).astype(int)

            lv_volume = np.sum(lv_mask, axis = (0,1,2))
            rv_volume = np.sum(rv_mask, axis = (0,1,2))



In [ ]:
def read_dicom_from_orthanc(orthanc_id, ORTHANC, AUTH):
    instances = requests.get(f"{ORTHANC}/series/{orthanc_id}/instances",auth=AUTH).json()
    ds_list = []
    for inst in instances:
        instance_id = inst["ID"]
        r = requests.get(f"{ORTHANC}/instances/{instance_id}/file", auth=AUTH)
        ds = dicom.dcmread(io.BytesIO(r.content))
        ds_list.append(ds)

    return ds_list


def send_series_to_orthanc(new_dcm, old_dcm, ORTHANC, AUTH):
    new_series_uid = generate_uid()
    for i in range(len(new_dcm)):
        old_dcm[i].SeriesInstanceUID = new_series_uid
        old_dcm[i].SeriesDescription = 'Roundel Processed Series'
        old_dcm[i].PixelData = new_dcm[i].astype(np.uint16).tobytes()
        old_dcm[i].SOPInstanceUID = generate_uid()
        old_dcm[i].file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
        old_dcm[i].is_little_endian = True
        old_dcm[i].is_implicit_VR = False
        buffer = io.BytesIO()
        old_dcm[i].save_as(buffer)
        buffer.seek(0)
        upload = requests.post(f"{ORTHANC}/instances",data=buffer.read(),auth=AUTH,headers={"Content-Type": "application/dicom", "Expect": ""})
        upload.raise_for_status()
    return new_series_uid

def get_orthanc_series_data_from_uid(series_uid, ORTHANC, AUTH):
    payload = {
        "Level": "Series",
        "Expand": True,
        "Query": {
            "SeriesInstanceUID": series_uid
        }
    }

    r = requests.post(f"{ORTHANC}/tools/find", json=payload, auth=AUTH)
    r.raise_for_status()
    results = r.json()

    if not results:
        return None

    return results[0]

def get_volumes(image,mask):
    new_im_all = []
    LV_vol = []
    RV_vol = []
    LV_mass = []
    RV_mass = []
    trigger_time = []
    pixel_spacing = image[0].PixelSpacing
    slice_thickness = image[0].SpacingBetweenSlices
    volume = (pixel_spacing[0]*pixel_spacing[1]*slice_thickness)/1000
    for i in range(len(image)):
        mask_dat = mask[i].pixel_array
        mask_bin = mask_dat>0
        im_dat = image[i].pixel_array
        new_im = im_dat * mask_bin
        new_im_all.append(new_im)
        trigger_time.append(image[i].InstanceNumber)
        
        LV_vol.append(np.sum(mask_dat==500)*volume)
        RV_vol.append(np.sum(mask_dat==1000)*volume)
        LV_mass.append(np.sum(mask_dat==1500)*volume)
        RV_mass.append(np.sum(mask_dat==2000)*volume)

    new_orthanc_id = send_series_to_orthanc(new_im_all, image, ORTHANC = "http://localhost:8042", AUTH = ("orthanc","orthanc"))
    return new_orthanc_id, LV_vol, RV_vol, LV_mass, RV_mass, trigger_time

In [ ]:
for orthanc_study_id in study_ids_not_roundel:
    study = db.get(Study.orthanc_study_id == orthanc_study_id)
    print(study["patient_name"])
    new_series_records = []
    all_dcms = []
    LV_volumes = []
    LV_masses = []
    RV_volumes = []
    RV_masses = []
    tt = []

    for s in study["series"]:
        if s.get('series_description', False)=='DL Processed':
            mask_for_roundel = read_dicom_from_orthanc(s['orthanc_series_id'], ORTHANC, AUTH)
            image_for_roundel = read_dicom_from_orthanc(s['associated_orthanc_id'], ORTHANC, AUTH)
            
            
            new_im_all, LV_vol, RV_vol, LV_mass, RV_mass, trig_times = get_volumes(image_for_roundel,mask_for_roundel)

            LV_volumes.append(LV_vol)
            RV_volumes.append(RV_vol)
            all_dcms.append(new_im_all)
            tt.append(trig_times)
            
    print(s.get('series_description'))


    
    
        

NameError: name 'study_ids_not_roundel' is not defined

In [ ]:
LV_volumes[0]

In [ ]:
import numpy as np

values = np.array(LV_volumes[0])
labels = np.array(tt[0])

idx = np.argsort(labels)

labels_sorted = labels[idx]
values_sorted = values[idx]

plt.plot(values_sorted)
    